In [1]:
# Obtain the final graph dataset that the model is going to be trained with
# This dataset consists of 6 node level features, 1 edge level feature, 1 graph level feature and the ground truth
# Graph: nodes (COM coordinates) and edges (method Leon)
# 6 node level features: cell (area / perimeter / shape index), nucleus (area / perimeter / shape index)
# 1 edge level feature: edge euclidean length
# 1 graph level feature: nucleus area fraction Phi
# Ground truth: diffusion constant D

In [3]:
import h5py
import numpy as np
import os
import matplotlib as mpl
import matplotlib.pyplot as plt
from fftmsd import calc_msd_fft, unwrap, calc_acf
from pathlib import Path
from cell2image.image import read_h5_lattice_snapshots
from cell2image.shape import (
    get_perimeter_estimator,
    calc_area,
    calc_shape_index)
from cell2image import image as cimg
import networkx as nx
import pickle
import torch
import scipy.io
from torch_geometric.data import Dataset, Data
from scipy.spatial import Delaunay, Voronoi
from shapely.geometry import Polygon

/home/20234017/.local/lib/python3.11/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: /vast.mnt/home/20234017/.local/lib/python3.11/site-packages/torch_scatter/_version_cuda.so: undefined symbol: _ZN5torch3jit17parseSchemaOrNameERKSs
  import torch_geometric.typing


In [4]:
# Check if GPU is available

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

True
Tesla P100-PCIE-16GB
cuda


In [5]:
# Function to compute diffusion constants from msd on the real MCS time steps

def compute_diffusion_from_msd(msd, time_step, d, time_boundary=1e5):
    """
    msd: individual msd trajectories
    time_step: the time interval in between generation of data of com (in this case 100)
    d: dimension of the problem (in this case d=2)
    time_boundary: time value that is the boundary for calculating the diffusion constant
    """
    t = np.arange(len(msd)) * time_step
    mask = t >= time_boundary
    slope = np.polyfit(t[mask], msd[mask], 1)[0]
    D = slope / (2 * d) 
    return D, t

In [6]:
# Function to get the adjacency of the nodes

def get_adjacency_from_labelled_image(img: np.ndarray):
    cell_ids = np.unique(img)
    adjacency = {cell_id: cimg.get_cell_neighbour_ids(img, cell_id) for cell_id in cell_ids}
    return adjacency

In [7]:
# Function to obtain a list of edge tuples from the adjacency

def edges_from_adj(adj: dict[int, set[int]]) -> set[tuple[int, int]]:
    edges = set()
    for node, neighbors in adj.items():
        for neighbor in neighbors:
            if node < neighbor:
                edges.add((node, neighbor))

    return edges

In [8]:
# Function to compute the features from a snapshot, handle 1 snapshot at a time, independently 

def GNN_dataset(com_coordinates, cell_ids, cell_types, cluster_ids, D):

    # Create list of nodes
    node_sublist = com_coordinates
    
    # Edge construction
    edges = set()
    adjacency = get_adjacency_from_labelled_image(cluster_ids)
    edges_set = edges_from_adj(adjacency)
    edges.update(edges_set)
    
    # Determine area, perimeter and shape index of all cells
    perimeter_estimator = get_perimeter_estimator()
    
    nucleus_areas = np.array([])
    nucleus_perimeters = np.array([])
    nucleus_shape_indices = np.array([])
    for cell_id in range(200):
        if cell_id % 2 == 1:
            nucleus_areas = np.append(nucleus_areas, calc_area(cell_ids, cell_id))
            nucleus_perimeters = np.append(nucleus_perimeters, perimeter_estimator(cell_ids, cell_id))
            nucleus_shape_indices = np.append(nucleus_shape_indices, calc_shape_index(cell_ids, cell_id, perimeter_estimator))
    
    cell_areas = np.array([])
    cell_perimeters = np.array([])
    cell_shape_indices = np.array([])
    for cell_id in range(100):
        cell_areas = np.append(cell_areas, calc_area(cluster_ids, cell_id))
        cell_perimeters = np.append(cell_perimeters, perimeter_estimator(cluster_ids, cell_id))
        cell_shape_indices = np.append(cell_shape_indices, calc_shape_index(cluster_ids, cell_id, perimeter_estimator))

    # Determine area fraction
    fraction_allcells = nucleus_areas / cell_areas
    area_frac = torch.tensor(np.mean(fraction_allcells), dtype=torch.float)

    # collect nodal and bulk information
    node_input = np.hstack((nucleus_areas.reshape(-1, 1), 
                            nucleus_perimeters.reshape(-1, 1), 
                            nucleus_shape_indices.reshape(-1, 1),
                            cell_areas.reshape(-1, 1), 
                            cell_perimeters.reshape(-1, 1), 
                            cell_shape_indices.reshape(-1, 1)))
    x = torch.tensor(node_input, dtype=torch.float) # All features together
    cell_pos = torch.tensor(node_sublist, dtype=torch.float)
    
    # ground truth
    diffConst = torch.tensor(D, dtype=torch.float)

    # get edge list
    edge_list = np.array(list(edges))
    edge_list = np.transpose(edge_list)
    
    edge_index = torch.tensor(edge_list, dtype=torch.long)
    
    edge_attr_all = np.sqrt(np.sum((node_sublist[edge_index[0, :]] - node_sublist[edge_index[1, :]])**2, axis=1))
    mask = edge_attr_all < 100
    edge_index = edge_index[:, mask]
    edge_attr = edge_attr_all[mask]
    edge_attr = torch.tensor(edge_attr, dtype=torch.float)

    cell_ids = torch.tensor(cell_ids, dtype=torch.float)

    # save data
    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, cell_pos = cell_pos,
                diffConst = diffConst, area_frac = area_frac, voronoi_equivalent = cell_ids)

    return data

In [9]:
# Function to get all the data ready to train and test the model with for 5x5 grid

def MainGetData_old(directory_old):
    
    msd = []
    D = []
    with h5py.File("msd_batch3.h5", "r") as f:
        for i in range(len(f.keys())):
            msd.append(f[f"msd_{i}"][:])
    for msd_i in msd:
        D.append(compute_diffusion_from_msd(msd_i, 100, 2)[0])
    D = np.array(D)
    D_avg = D.reshape(-1, 12).mean(axis=1)
    D_avg_expanded = np.repeat(D_avg, 12) # this uses only the averaged D values
        
    foldernames = []
    for foldername in sorted(
        [f for f in os.listdir(directory_old) if not f.startswith('.')],
        key=lambda x: int(x.replace("Simulation", ""))):
        foldernames.append(os.path.join(directory_old, foldername))
    print(f"The amount of folders in the old directory is:", len(foldernames))

    final_data_list = []
    for i, foldername in enumerate(foldernames):
        filenames = []
        for filename in os.listdir(foldername):
            if not filename.startswith('.'):
                filenames.append(os.path.join(foldername, filename))

        data_list = []
        # Load data (3 snapshots per simulation, index 50 of batch in lattice)
        # File 1 is COM, file 2 is configuration settings, file 3 is lattice
        with h5py.File(filenames[0], "r") as f:
            com_index = [15000, 55000, 95000] 
            data_cluster = f["simulation/cluster_com"][com_index, :, :] # gives (3, 200, 2)
            com_coordinates = data_cluster[:, ::2, :] % 300 # Shape (3, 100, 2) and wrapped within box size
    
        cell_ids = [] # All have shape (3, 300, 300) after data gets added
        cell_types = []
        cluster_ids = []
        with h5py.File(filenames[2], "r") as f:
            for batch_nr in [1,5,9]:
                g = f[f"batch_{batch_nr}"]
                cell_ids.append(g["cell_id"][50])
                cell_types.append(g["cell_type"][50])
                cluster_ids.append(g["cluster_id"][50])
        cell_ids = np.stack(cell_ids)
        cell_types = np.stack(cell_types)
        cluster_ids = np.stack(cluster_ids)

        for k in range(3):
            data = GNN_dataset(com_coordinates[k], 
                               cell_ids[k], 
                               cell_types[k], 
                               cluster_ids[k], 
                               D_avg_expanded[i]) 
            data_list.append(data)

        final_data_list.append(data_list)
    
    return final_data_list

In [10]:
# Function to get all the data ready to train and test the model with for expansion to 9x9 grid 

def MainGetData_new(directory_new, directory_corrected):
    
    msd = []
    D = []
    with h5py.File("msd_batch3_GE_[corrected].h5", "r") as f:
        for i in range(len(f.keys())):
            msd.append(f[f"msd_{i}"][:])
    for msd_i in msd:
        D.append(compute_diffusion_from_msd(msd_i, 100, 2)[0])
    D = np.array(D)
    D_avg = D.reshape(-1, 12).mean(axis=1)
    D_avg_expanded = np.repeat(D_avg, 12) # this uses only the averaged D values
        
    foldernames = []
    foldernames_GE_MC = []
    for foldername in sorted(
        [f for f in os.listdir(directory_new) if not f.startswith('.')],
        key=lambda x: int(x.replace("Simulation", ""))):
        foldernames.append(os.path.join(directory_new, foldername))
    print(f"The amount of folders in the GE directory is:", len(foldernames))
    for foldername in sorted(
        [f for f in os.listdir(directory_corrected) if not f.startswith('.')],
        key=lambda x: int(x.replace("Simulation", ""))):
        foldernames_GE_MC.append(os.path.join(directory_corrected, foldername))
    print(f"The amount of folders in the GE mistake correction directory is:", len(foldernames_GE_MC))

    foldernames[24:36]   = foldernames_GE_MC[0:12]
    foldernames[84:96]   = foldernames_GE_MC[12:24]
    foldernames[144:156] = foldernames_GE_MC[24:36]
    foldernames[204:216] = foldernames_GE_MC[36:48]

    print(f'Amount of folders after replacement {len(foldernames)}')

    final_data_list = []
    for i, foldername in enumerate(foldernames):
        filenames = []
        for filename in os.listdir(foldername):
            if not filename.startswith('.'):
                filenames.append(os.path.join(foldername, filename))

        data_list = []
        # Load data (3 snapshots per simulation, index 50 of each odd batch in lattice)
        # File 1 is COM, file 2 is configuration settings, file 3 is lattice
        with h5py.File(filenames[0], "r") as f:
            com_index = [15000, 55000, 95000] 
            data_cluster = f["simulation/cluster_com"][com_index, :, :] # gives (3, 200, 2)
            com_coordinates = data_cluster[:, ::2, :] % 300 # Shape (3, 100, 2) and wrapped within box size
    
        cell_ids = [] # All have shape (3, 300, 300) after data gets added
        cell_types = []
        cluster_ids = []
        with h5py.File(filenames[2], "r") as f:
            for batch_nr in [1,5,9]:
                g = f[f"batch_{batch_nr}"]
                cell_ids.append(g["cell_id"][50])
                cell_types.append(g["cell_type"][50])
                cluster_ids.append(g["cluster_id"][50])
        cell_ids = np.stack(cell_ids)
        cell_types = np.stack(cell_types)
        cluster_ids = np.stack(cluster_ids)

        for k in range(3):
            data = GNN_dataset(com_coordinates[k], 
                               cell_ids[k], 
                               cell_types[k], 
                               cluster_ids[k], 
                               D_avg_expanded[i]) 
            data_list.append(data)

        final_data_list.append(data_list)
    
    return final_data_list

In [11]:
# Obtain the data to train the model with from the HDF5 files - 5x5 BATCH

directory_old = f"scratch/Simulation_batch3"
data_model_old = MainGetData_old(directory_old)

The amount of folders in the old directory is: 300


In [13]:
# Obtain the data to train the model with from the HDF5 files - GRID EXPAND TO 9x9 BATCH

directory_new = f"scratch/Simulation_batch3_GE"
directory_corrected = f"scratch/Simulation_batch3_GE_mistakeCorrection"
data_model_new = MainGetData_new(directory_new, directory_corrected)

The amount of folders in the GE directory is: 672
The amount of folders in the GE mistake correction directory is: 48
Amount of folders after replacement 672


In [14]:
# Concatenate the data such that it is in order of the grid in case of train data
# For the validation data the oder does not matter much so just leave in unordered

data_model = data_model_old + data_model_new

order = [
    1,46,2,47,3,48,4,49,5,
    26,66,27,67,28,68,29,69,30,
    6,50,7,51,8,52,9,53,10,
    31,70,32,71,33,72,34,73,35,
    11,54,12,55,13,56,14,57,15,
    36,74,37,75,38,76,39,77,40,
    16,58,17,59,18,60,19,61,20,
    41,78,42,79,43,80,44,81,45,
    21,62,22,63,23,64,24,65,25]

grouped = [data_model[i:i+12] for i in range(0, len(data_model), 12)]

data_model_sorted = [
    item
    for index in order
    for item in grouped[index - 1]]

In [16]:
# Save the entire processed dataset in a pickle file

def save_dataset(data_list, file_name):
    with open(file_name, 'wb') as f:
        pickle.dump(data_list, f)

save_dataset(data_model, "AllProcessedData_BigGrid_batch3_[FINAL_VALIDATION].pkl")

In [17]:
# Check length of data

len(data_model)

<class 'list'>


972

In [18]:
# Visualize shape of data (all start with (972, 5) and after that the tensor cones)

sample = data_model[0][0]

print("x:", sample.x.shape) # Contains for 100 cells 6 features per cell
print("edge_index:", sample.edge_index.shape) # Contains node numbers of connected nodes (2 numbers) and 259 edges
print("edge_attr:", sample.edge_attr.shape) # Contains 259 edge lengths
print("cell_pos:", sample.cell_pos.shape) # Contains for 100 cells 2 coordinates (x and y)
print("diffConst:", sample.diffConst.shape) # Contains single number
print("area_frac:", sample.area_frac.shape) # Contains single number
print("voronoi_equivalent:", sample.voronoi_equivalent.shape) # Contains lattice of 300x300 pixels with each a cell ID

x: torch.Size([100, 6])
edge_index: torch.Size([2, 261])
edge_attr: torch.Size([261])
cell_pos: torch.Size([100, 2])
diffConst: torch.Size([])
area_frac: torch.Size([])
voronoi_equivalent: torch.Size([300, 300])


In [20]:
# Visualize the 81 different diffusion constants in the data
for i in range(0, 972, 12):
    sample2 = data_model[i][2]
    print("diffConst:", sample2.diffConst)

diffConst: tensor(0.0473)
diffConst: tensor(0.1108)
diffConst: tensor(1.3310)
diffConst: tensor(3.0811)
diffConst: tensor(5.5592)
diffConst: tensor(0.0257)
diffConst: tensor(0.0855)
diffConst: tensor(1.1599)
diffConst: tensor(2.7115)
diffConst: tensor(5.0023)
diffConst: tensor(0.0414)
diffConst: tensor(0.0638)
diffConst: tensor(0.6394)
diffConst: tensor(1.9041)
diffConst: tensor(3.8089)
diffConst: tensor(0.0285)
diffConst: tensor(0.0368)
diffConst: tensor(0.1101)
diffConst: tensor(0.9926)
diffConst: tensor(2.2323)
diffConst: tensor(0.0356)
diffConst: tensor(0.0123)
diffConst: tensor(0.0235)
diffConst: tensor(0.0514)
diffConst: tensor(0.4688)
diffConst: tensor(0.0211)
diffConst: tensor(0.1370)
diffConst: tensor(1.1739)
diffConst: tensor(2.7949)
diffConst: tensor(5.7892)
diffConst: tensor(0.0476)
diffConst: tensor(0.1610)
diffConst: tensor(0.8891)
diffConst: tensor(2.3853)
diffConst: tensor(4.3748)
diffConst: tensor(0.0162)
diffConst: tensor(0.0939)
diffConst: tensor(0.4834)
diffConst: t